In [1]:
!pip install feast pandas pyarrow -q

!mkdir -p /content/feast_paysim/feature_repo/data

%cd /content/feast_paysim/feature_repo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.1/513.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
/content/fea

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
data_size = 100
customer_ids = [f"C{i}" for i in np.random.randint(1000, 1020, size=data_size)]
now = datetime.utcnow()
timestamps = [now - timedelta(days=int(np.random.randint(1, 5)), hours=int(np.random.randint(1, 24))) for _ in range(data_size)]

data = {
    "nameOrig": customer_ids,
    "amount": np.random.uniform(10, 5000, size=data_size).astype(np.float32),
    "oldbalanceOrg": np.random.uniform(100, 10000, size=data_size).astype(np.float32),
    "newbalanceOrig": np.random.uniform(0, 5000, size=data_size).astype(np.float32),
    "transaction_count": np.random.randint(1, 10, size=data_size).astype(np.int64),
    "event_timestamp": timestamps,
    "created_timestamp": [now] * data_size
}

df = pd.DataFrame(data)
df.to_parquet("data/paysim_features.parquet")


/tmp/ipykernel_8875/1756050913.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


In [7]:
yaml_content = """
project: paysim_fraud
registry: data/registry.db
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
offline_store:
    type: file
"""
with open("feature_store.yaml", "w") as f:
    f.write(yaml_content.strip())

definitions_content = """
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

paysim_source = FileSource(
    path="data/paysim_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

customer = Entity(name="customer_id", join_keys=["nameOrig"])

customer_stats_view = FeatureView(
    name="customer_transaction_stats",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="amount", dtype=Float32),
        Field(name="oldbalanceOrg", dtype=Float32),
        Field(name="newbalanceOrig", dtype=Float32),
        Field(name="transaction_count", dtype=Int64),
    ],
    online=True,
    source=paysim_source,
)
"""
with open("definitions.py", "w") as f:
    f.write(definitions_content.strip())



In [8]:
!feast apply

!feast materialize-incremental 2026-05-30T23:59:59

/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'reset

In [9]:
from feast import FeatureStore
import pandas as pd
from datetime import datetime

store = FeatureStore(repo_path=".")

print("--- 1. OFFLINE STORE SORĞUSU (Model Training üçün) ---")
entity_df = pd.DataFrame.from_dict({
    "nameOrig": ["C1001", "C1005"],
    "event_timestamp": [datetime.utcnow(), datetime.utcnow()]
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_transaction_stats:amount",
        "customer_transaction_stats:transaction_count",
    ],
).to_df()
print(training_df)

print("\n" + "="*50 + "\n")

print("--- 2. ONLINE STORE SORĞUSU (Canlı Təxmin/Inference üçün) ---")
online_features = store.get_online_features(
    features=[
        "customer_transaction_stats:amount",
        "customer_transaction_stats:transaction_count",
    ],
    entity_rows=[{"nameOrig": "C1001"}],
).to_dict()
print(online_features)

--- 1. OFFLINE STORE SORĞUSU (Model Training üçün) ---
  nameOrig                  event_timestamp       amount  transaction_count
0    C1005 2026-05-28 21:32:15.927092+00:00  3607.490234                  1
1    C1001 2026-05-28 21:32:15.927084+00:00  1547.223389                  2


--- 2. ONLINE STORE SORĞUSU (Canlı Təxmin/Inference üçün) ---
{'nameOrig': ['C1001'], 'amount': [1547.223388671875], 'transaction_count': [2]}


/tmp/ipykernel_8875/2264365628.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "event_timestamp": [datetime.utcnow(), datetime.utcnow()]
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
!apt-get install redis-server -y -qq > /dev/null
!redis-server --daemonize yes

!redis-cli ping

yaml_content = """
project: paysim_fraud
registry: data/registry.db
provider: local
online_store:
    type: redis
    connection_string: redis://localhost:6379/0
offline_store:
    type: file
"""

%cd /content/feast_paysim/feature_repo
with open("feature_store.yaml", "w") as f:
    f.write(yaml_content.strip())

print("Redis uğurla başladıldı və Feast konfiqurasiyasına inteqrasiya olundu!")

PONG
/content/feast_paysim/feature_repo
Redis uğurla başladıldı və Feast konfiqurasiyasına inteqrasiya olundu!


In [13]:
yaml_content = """
project: paysim_fraud
registry: data/registry.db
provider: local
online_store:
    type: redis
    connection_string: localhost:6379
offline_store:
    type: file
"""

%cd /content/feast_paysim/feature_repo
with open("feature_store.yaml", "w") as f:
    f.write(yaml_content.strip())

print("1. Yaml faylı düzgün formatda yenidən yazıldı.")

!rm -f data/registry.db

print("\n2. Feast apply işə düşür...")
!feast apply

print("\n3. Redis-ə materializasiya prosesi başlayır...")
!feast materialize-incremental 2026-05-30T23:59:59

/content/feast_paysim/feature_repo
1. Yaml faylı düzgün formatda yenidən yazıldı.

2. Feast apply işə düşür...
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()
/usr/local/lib/python3.12/dist-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 

In [14]:
from feast import FeatureStore
import pandas as pd
from datetime import datetime

store = FeatureStore(repo_path=".")

print("--- 1. OFFLINE STORE SORĞUSU (Parquet-dən Model Training üçün) ---")
entity_df = pd.DataFrame.from_dict({
    "nameOrig": ["C1001", "C1005"],
    "event_timestamp": [datetime.utcnow(), datetime.utcnow()]
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_transaction_stats:amount",
        "customer_transaction_stats:transaction_count",
    ],
).to_df()
print(training_df)

print("\n" + "="*50 + "\n")

print("--- 2. ONLINE STORE SORĞUSU (Redis-dən Canlı Təxmin/Inference üçün) ---")
online_features = store.get_online_features(
    features=[
        "customer_transaction_stats:amount",
        "customer_transaction_stats:transaction_count",
    ],
    entity_rows=[{"nameOrig": "C1001"}],
).to_dict()
print(online_features)

--- 1. OFFLINE STORE SORĞUSU (Parquet-dən Model Training üçün) ---


/tmp/ipykernel_8875/3170650745.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "event_timestamp": [datetime.utcnow(), datetime.utcnow()]
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  nameOrig                  event_timestamp       amount  transaction_count
0    C1005 2026-05-28 22:39:48.998793+00:00  3607.490234                  1
1    C1001 2026-05-28 22:39:48.998786+00:00  1547.223389                  2


--- 2. ONLINE STORE SORĞUSU (Redis-dən Canlı Təxmin/Inference üçün) ---
{'nameOrig': ['C1001'], 'amount': [1547.223388671875], 'transaction_count': [2]}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
note_content = """
# Feature Store Design and Trade-offs (PaySim Fraud Project)

## 1. Architecture Overview
- **Offline Store:** Apache Parquet format (`data/paysim_features.parquet`). Used for batch storage, data analysis, and point-in-time historical queries for ML model training.
- **Online Store:** Redis In-Memory Database (`redis://localhost:6379`). Used for low-latency, ultra-fast online feature serving during inference.
- **Feature Store Framework:** Feast (Feature Store for Machine Learning).

## 2. Defined Entities & Features
- **Entity:** `customer_id` (mapped via `nameOrig` join key).
- **Feature View:** `customer_transaction_stats`
  - `amount` (Float32)
  - `oldbalanceOrg` (Float32)
  - `newbalanceOrig` (Float32)
  - `transaction_count` (Int64)

## 3. Core Trade-offs & Engineering Decisions
- **Storage Cost vs. Latency:** Moving from SQLite to Redis increases operational/infrastructure costs, but drastically reduces online inference latency from milliseconds to microseconds, which is a hard requirement for real-time banking fraud detection.
- **Feature Freshness (TTL):** Managed with a 30-day Time-To-Live (TTL) policy. This limits storage footprint in Redis and ensures old, stale behavioral data does not negatively impact model accuracy.
- **Consistency:** Feast guarantees that the exact same feature engineering logic is applied during training (Offline) and serving (Online), eliminating the notorious "Training-Serving Skew".
"""

with open("/content/feast_paysim/feature_repo/note.md", "w") as f:
    f.write(note_content.strip())

